In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
food_df = pd.read_csv("abbrev_cleaned.csv")

print(food_df.head())

print(food_df.info())

                  food_name  calories  protein    fat  carbs  fiber  sugar
0          Butter,with salt       717     0.85  81.11   0.06    0.0   0.06
1  Butter,whipped,with salt       717     0.85  81.11   0.06    0.0   0.06
2      Butter oil,anhydrous       876     0.28  99.48   0.00    0.0   0.00
3               Cheese,blue       353    21.40  28.74   2.34    0.0   0.50
4              Cheese,brick       371    23.24  29.68   2.79    0.0   0.51
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8618 entries, 0 to 8617
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   food_name  8618 non-null   object 
 1   calories   8618 non-null   int64  
 2   protein    8618 non-null   float64
 3   fat        8618 non-null   float64
 4   carbs      8618 non-null   float64
 5   fiber      8618 non-null   float64
 6   sugar      8618 non-null   float64
dtypes: float64(5), int64(1), object(1)
memory usage: 471.4+ KB
None


In [ ]:
food_features = food_df.select_dtypes(
    include=[np.number]
)

In [ ]:
food_features = food_features.fillna(0)

In [ ]:
scaler = StandardScaler()

food_scaled = scaler.fit_transform(
    food_features
)

In [ ]:
X_train, X_test = train_test_split(
    food_scaled,
    test_size=0.2,
    random_state=42
)

In [ ]:
def cosine_loss(y_true, y_pred):
    y_true = tf.math.l2_normalize(
        y_true,
        axis=1
    )

    y_pred = tf.math.l2_normalize(
        y_pred,
        axis=1
    )

    similarity = tf.reduce_sum(
        y_true * y_pred,
        axis=1
    )

    return 1 - tf.reduce_mean(
        similarity
    )

In [ ]:
class CustomLogger(
    tf.keras.callbacks.Callback
):
    def on_epoch_end(
        self,
        epoch,
        logs=None
    ):

        print(
            f"Epoch {epoch + 1} | "
            f"Loss: {logs['loss']:.4f} | "
            f"Val Loss: {logs['val_loss']:.4f}"
        )

In [ ]:
input_dim = X_train.shape[1]

In [ ]:
inputs = layers.Input(
    shape=(input_dim,)
)

In [ ]:
x = layers.Dense(
    256,
    activation='relu'
)(inputs)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

In [ ]:
x = layers.Dense(
    128,
    activation='relu'
)(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

In [ ]:
x = layers.Dense(
    64,
    activation='relu'
)(x)

In [ ]:
embedding = layers.Dense(
    32,
    activation='linear',
    name="food_embedding"
)(x)

In [ ]:
outputs = layers.Dense(
    input_dim,
    activation='linear'
)(embedding)

In [ ]:
model = Model(
    inputs,
    outputs
)

In [ ]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss=cosine_loss,

    metrics=['mae']

)

In [ ]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ food_embedding (Dense)          │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,758 (182.65 KB)

 Trainable params: 45,990 (179.65 KB)

 Non-trainable params: 768 (3.00 KB)

In [ ]:
history = model.fit(
    X_train,
    X_train,
    validation_data=(
        X_test,
        X_test
    ),

    epochs=20,
    batch_size=32,
    callbacks=[
        CustomLogger()
    ]

)

Epoch 1/20
204/216 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1330 - mae: 2.1620Epoch 1 | Loss: 0.0631 | Val Loss: 0.0175
216/216 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0631 - mae: 2.7201 - val_loss: 0.0175 - val_mae: 0.9076
Epoch 2/20
208/216 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0273 - mae: 3.6183Epoch 2 | Loss: 0.0255 | Val Loss: 0.0087
216/216 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0255 - mae: 3.8743 - val_loss: 0.0087 - val_mae: 3.0639
Epoch 3/20
209/216 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0217 - mae: 4.4925Epoch 3 | Loss: 0.0213 | Val Loss: 0.0054
216/216 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0213 - mae: 4.6245 - val_loss: 0.0054 - val_mae: 4.6206
Epoch 4/20
212/216 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0189 - mae: 5.0280Epoch 4 | Loss: 0.0183 | Val Loss: 0.0044
216/216 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0183 - mae: 5.1689 - val_loss: 0.0044 - val_mae: 5.4664
Epoch 5/20
208/216 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0193 - mae: 5.4931Epoc

In [ ]:
model.save(
    "saved_model/food_recommender.keras"
)

In [ ]:
embedding_model = Model(
    inputs=model.input,
    outputs=model.get_layer(
        "food_embedding"
    ).output
)

In [ ]:
food_embeddings = embedding_model.predict(
    food_scaled
)

270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [ ]:
def filter_by_goal(df, goal):

    filtered_df = df.copy()
    if goal == "weight_loss":
        filtered_df = filtered_df[
            filtered_df["calories"] <
            filtered_df["calories"].median()
        ]

    elif goal == "muscle_gain":
        filtered_df = filtered_df[
            filtered_df["protein"] >
            filtered_df["protein"].median()
        ]

    elif goal == "maintain":
        filtered_df = filtered_df

    return filtered_df

In [ ]:
def recommend_food_by_goal(
    food_index,
    goal="maintain",
    top_n=5
):

    filtered_df = filter_by_goal(
        food_df,
        goal
    )

    filtered_indices = filtered_df.index.tolist()
    filtered_embeddings = food_embeddings[
        filtered_indices
    ]

    target_vector = food_embeddings[
        food_index
    ].reshape(1, -1)

    similarities = cosine_similarity(
        target_vector,
        filtered_embeddings
    )[0]

    top_indices = similarities.argsort()[
        -top_n:
    ][::-1]

    recommended_indices = [
        filtered_indices[i]
        for i in top_indices
    ]

    return food_df.iloc[
        recommended_indices
    ]

In [ ]:
recommend_food_by_goal(
    food_index=10,
    goal="muscle_gain",
    top_n=5
)

,food_name,calories,protein,fat,carbs,fiber,sugar
10,"Cheese,colby",394,23.76,32.11,2.57,0.0,0.52
148,"Cheese,low-sodium,cheddar or colby",398,24.35,32.62,1.91,0.0,0.49
8,"Cheese,cheddar",406,24.04,33.82,1.33,0.0,0.28
224,"Cheese,cheddar,sharp,sliced",410,24.25,33.82,2.13,0.0,0.27
9,"Cheese,cheshire",387,23.37,30.60,4.78,0.0,0.00


In [ ]:
train_mae = history.history['mae'][-1]
val_mae = history.history['val_mae'][-1]

In [ ]:
summary = pd.DataFrame({
    "Model": ["Food Recommender DL"],
    "Train MAE": [train_mae],
    "Validation MAE": [val_mae]
})

display(summary)

,Model,Train MAE,Validation MAE
0,Food Recommender DL,9.741502,9.824799
